## Fengyun-2D satellite

In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from xgboost.callback import EarlyStopping
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import timedelta

### Load and preprocess the TLE data


In [2]:
import pandas as pd
# Replace with appropriate path 
df_tles = pd.read_csv('/Users/kurinjiarivazhagan/Desktop/trimester 5/satellite_data/orbital_elements/Fengyun-2D.csv', 
                      index_col=0, parse_dates=True)

# Check if datetime index is timezone-naive or timezone-aware
if df_tles.index.tz is None:
    df_tles.index = df_tles.index.tz_localize('UTC')
else:
    df_tles.index = df_tles.index.tz_convert('UTC')


print(df_tles.describe())
print(df_tles.index.inferred_type)



       eccentricity  argument of perigee  inclination  mean anomaly  \
count   1187.000000          1187.000000  1187.000000   1187.000000   
mean       0.000159             3.142386     0.035877     -3.193376   
std        0.000071             2.304360     0.008510      1.465265   
min        0.000006             0.000339     0.020778     -6.272115   
25%        0.000115             0.932269     0.028463     -4.295577   
50%        0.000160             2.339151     0.036228     -3.237645   
75%        0.000199             5.722917     0.043127     -2.122498   
max        0.000888             6.281613     0.051880     -0.006215   

       Brouwer mean motion  right ascension  
count         1.187000e+03      1187.000000  
mean          4.375016e-03         1.292740  
std           2.366158e-07         0.073150  
min           4.374399e-03         1.195585  
25%           4.374820e-03         1.235582  
50%           4.375012e-03         1.269402  
75%           4.375212e-03         1.3

### Extract and scale Brouwer mean motion

In [3]:
df_element_1 = df_tles[["Brouwer mean motion"]]
df_element_1 = (df_element_1 - df_element_1.mean())*1e7
df_element_1.describe()


,Brouwer mean motion
count,1.187000e+03
mean,7.117190e-12
std,2.366158e+00
min,-6.162539e+00
25%,-1.961845e+00
50%,-4.135635e-02
75%,1.966321e+00
max,8.731502e+00


### Visualize the scaled element

In [4]:
import plotly.graph_objects as go


fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_element_1.index, 
    y=df_element_1[df_element_1.columns[0]],  
    mode="lines",  
    name="Orbital Element"
))

fig.update_layout(
    title="Brouwer mean motion Over Time",
    xaxis_title="Time",
    yaxis_title="Brouwer mean motion",
    template="plotly",
    width=1400
)

fig.show()


### Create lag features for time series forecasting

In [5]:
NUM_LAG_FEATURES = 3

df_y = df_element_1.copy()
df_x = df_element_1.shift(1).rename(columns={"Brouwer mean motion": "bmm_lag_1"})

for lag in range(2, NUM_LAG_FEATURES + 1):
    df_x[f"bmm_lag_{lag}"] = df_element_1.shift(lag)

# Drop rows with NaNs
df_x = df_x.iloc[NUM_LAG_FEATURES:]
df_y = df_y.iloc[NUM_LAG_FEATURES:]


### Split for hyperparameter tuning


In [16]:
# Split for tuning
split_index = int(len(df_x) * 0.8)
split_date = df_x.index[split_index].strftime("%Y-%m-%d")

df_x_train = df_x[:split_date]
df_y_train = df_y[:split_date]
df_x_test = df_x[split_date:]
df_y_test = df_y[split_date:]

# Tune XGBoost model with early stopping
tuned_model = XGBRegressor(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.1,
    objective="reg:squarederror",
    eval_metric="rmse",
    early_stopping_rounds=10,
    random_state=42
)

tuned_model.fit(
    df_x_train,
    df_y_train.values.ravel(),
    eval_set=[(df_x_train, df_y_train), (df_x_test, df_y_test)],
    verbose=True
)

# View best iteration and RMSE
best_n_estimators = tuned_model.best_iteration + 1  
print(f"Best number of trees: {best_n_estimators}")


[0]	validation_0-rmse:2.14304	validation_1-rmse:2.33225
[1]	validation_0-rmse:1.98042	validation_1-rmse:2.17859
[2]	validation_0-rmse:1.83792	validation_1-rmse:2.04577
[3]	validation_0-rmse:1.71326	validation_1-rmse:1.93249
[4]	validation_0-rmse:1.60381	validation_1-rmse:1.83629
[5]	validation_0-rmse:1.50906	validation_1-rmse:1.75241
[6]	validation_0-rmse:1.42727	validation_1-rmse:1.68253
[7]	validation_0-rmse:1.35648	validation_1-rmse:1.62464
[8]	validation_0-rmse:1.29590	validation_1-rmse:1.57298
[9]	validation_0-rmse:1.24408	validation_1-rmse:1.52972
[10]	validation_0-rmse:1.20005	validation_1-rmse:1.49592
[11]	validation_0-rmse:1.16256	validation_1-rmse:1.46584
[12]	validation_0-rmse:1.13100	validation_1-rmse:1.44430
[13]	validation_0-rmse:1.10443	validation_1-rmse:1.42424
[14]	validation_0-rmse:1.08168	validation_1-rmse:1.40941
[15]	validation_0-rmse:1.06202	validation_1-rmse:1.39635
[16]	validation_0-rmse:1.04571	validation_1-rmse:1.38768
[17]	validation_0-rmse:1.03150	validation


### Retrain final model on full dataset

In [7]:
# Use full data for final model
df_x_full = df_x.copy()
df_y_full = df_y.copy()

final_model = XGBRegressor(
    n_estimators=best_n_estimators,
    max_depth=3,
    learning_rate=0.1,
    objective="reg:squarederror",
    eval_metric="rmse",
    random_state=42
)

final_model.fit(df_x_full, df_y_full.values.ravel())


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=30,
             n_jobs=None, num_parallel_tree=None, ...)

### Make predictions on full data and compute residuals

In [8]:
# Predict and calculate residuals
y_pred_full = final_model.predict(df_x_full)
residuals_full = y_pred_full - df_y_full["Brouwer mean motion"].values

df_result = df_y_full.copy()
df_result["predicted"] = y_pred_full
df_result["residuals"] = residuals_full


###  Plot Observed vs Predicted (Full Time Range)

In [9]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["Brouwer mean motion"],
    mode='lines',
    name='Observed'
))

fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["predicted"],
    mode='lines',
    name='Predicted'
))

fig.update_layout(
    title='Observed vs Predicted Brouwer Mean Motion (Full Series)',
    xaxis_title='Time',
    yaxis_title='Brouwer Mean Motion (scaled)',
    template='plotly_white',
    width=1200
)

fig.show()


### Plot residuals over time 

In [10]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["residuals"],
    mode='lines+markers',
    name='Residuals'
))

fig.add_hline(y=0, line_dash="dot", line_color="black")

fig.update_layout(
    title='Residuals Over Time',
    xaxis_title='Time',
    yaxis_title='Residual (Observed - Predicted)',
    template='plotly_white',
    width= 1400
)

fig.show()


### Plot Residuals with Ground Truth Maneuvers

In [11]:
# Load maneuver data
ground_truth_df = pd.read_csv(
    "/Users/kurinjiarivazhagan/Desktop/trimester 5/satellite_data/orbital_elements/cleaned maneuver file/cleaned_FENGYUN-2D.csv",
    parse_dates=["Start_Timestamp", "End_Timestamp"]
)

for col in ["Start_Timestamp", "End_Timestamp"]:
    if ground_truth_df[col].dt.tz is None:
        ground_truth_df[col] = ground_truth_df[col].dt.tz_localize("UTC")
    else:
        ground_truth_df[col] = ground_truth_df[col].dt.tz_convert("UTC")


# Filter to full range
test_start = df_result.index.min()
test_end = df_result.index.max()

ground_truth_df_test = ground_truth_df[
    (ground_truth_df["Start_Timestamp"] >= test_start) &
    (ground_truth_df["Start_Timestamp"] <= test_end)
]

# Get residual range
y_min = df_result["residuals"].min()
y_max = df_result["residuals"].max()

# Plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["residuals"],
    mode='lines+markers',
    name='Residuals',
    marker=dict(size=4),
    hovertemplate='Date: %{x|%Y-%m-%d}<br>Residual: %{y:.4f}'
))

fig.add_hline(
    y=0,
    line_dash="dot",
    line_color="black",
    annotation_text="Zero Residual",
    annotation_position="top left"
)

# Ground truth maneuver lines
for _, row in ground_truth_df_test.iterrows():
    maneuver_date = row["Start_Timestamp"]
    hover_text = f"Maneuver Date: {maneuver_date.strftime('%Y-%m-%d')}"
    fig.add_trace(go.Scatter(
        x=[maneuver_date, maneuver_date],
        y=[y_min, y_max],
        mode='lines',
        line=dict(color='green', dash='dash'),
        name='Ground Truth Maneuver',
        hoverinfo='text',
        text=[hover_text, hover_text],
        showlegend=False
    ))

fig.add_trace(go.Scatter(
    x=[None],
    y=[None],
    mode='lines',
    line=dict(color='green', dash='dash'),
    name='Ground Truth Maneuver'
))

fig.update_layout(
    title='Residuals with Ground Truth Maneuvers',
    xaxis_title='Time',
    yaxis_title='Residual (Observed - Predicted)',
    template='plotly_white',
    width=1400,
    height=600,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig.show()


### Detect Anomalies (Using 3σ Rule)

 You're detecting anomalies by looking at how far the prediction errors (residuals) deviate from what's considered normal. First, you calculate the average residual and how much they typically vary (standard deviation). Then, you set a threshold range — anything beyond 3 standard deviations above or below the mean is flagged as an anomaly. This is because such extreme values are very rare in normal behavior. So, if the model makes a prediction that’s way off from what’s expected, it’s marked as an anomaly and the timestamp is recorded.

In [12]:

# Calculate mean and standard deviation of residuals
residual_mean = df_result["residuals"].mean()
residual_std = df_result["residuals"].std()
# Define upper and lower thresholds
upper = residual_mean + 3 * residual_std
lower = residual_mean - 3 * residual_std
# Flag anomalies: residuals that fall outside the threshold range
df_result["anomaly"] = (df_result["residuals"] > upper) | (df_result["residuals"] < lower)
print(f"Anomaly threshold range: {lower:.4f} to {upper:.4f}")
print(" Anomaly timestamps:")
print(df_result[df_result["anomaly"]].index)


Anomaly threshold range: -3.0582 to 3.0557
 Anomaly timestamps:
DatetimeIndex(['2011-03-29 17:14:06.382752+00:00',
               '2011-06-04 14:51:53.593919+00:00',
               '2011-08-06 00:13:33.024000+00:00',
               '2011-10-17 23:26:25.381824+00:00',
               '2012-05-17 23:53:05.038943+00:00',
               '2012-08-12 15:12:50.961888+00:00',
               '2012-11-01 22:03:07.395839+00:00',
               '2013-01-30 14:25:58.977119+00:00',
               '2013-02-06 20:31:07.342176+00:00',
               '2013-04-10 20:11:29.483807+00:00',
               '2013-06-27 01:38:55.516703+00:00',
               '2013-09-17 23:02:19.679135+00:00',
               '2013-11-23 00:32:53.214432+00:00',
               '2014-02-22 18:59:52.217376+00:00',
               '2014-05-09 21:31:10.111584+00:00',
               '2014-08-04 22:34:03.304991+00:00',
               '2014-08-06 01:58:28.565183+00:00',
               '2014-10-27 21:18:22.591007+00:00',
               '20

### Plotting detected anomalies vs ground truth

In [13]:

# Load ground truth maneuver data 
ground_truth_df = pd.read_csv(
    "/Users/kurinjiarivazhagan/Desktop/trimester 5/satellite_data/orbital_elements/cleaned maneuver file/cleaned_FENGYUN-2D.csv",
    parse_dates=["Start_Timestamp", "End_Timestamp"]
)

# Filter maneuver data to match df_result range 
test_start = df_result.index.min()
test_end = df_result.index.max()

ground_truth_df_test = ground_truth_df[
    (ground_truth_df["Start_Timestamp"] >= test_start) &
    (ground_truth_df["Start_Timestamp"] <= test_end)
]

# Get residual y-axis range for drawing vertical maneuver lines 
residuals_min = df_result["residuals"].min()
residuals_max = df_result["residuals"].max()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["Brouwer mean motion"],
    mode='lines+markers',
    name='Observed',
    marker=dict(size=4),
), secondary_y=False)

# Plot predicted values 
fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["predicted"],
    mode='lines+markers',
    name='Predicted',
    marker=dict(size=4),
), secondary_y=False)

# Plot residuals 
fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["residuals"],
    mode='lines+markers',
    name='Residuals',
    marker=dict(size=4),
), secondary_y=True)

# Plot detected anomalies 
fig.add_trace(go.Scatter(
    x=df_result[df_result["anomaly"]].index,
    y=df_result[df_result["anomaly"]]["residuals"],
    mode='markers',
    name='Detected Anomalies',
    marker=dict(color='red', size=10, symbol='circle'),
    hovertemplate='Anomaly Date: %{x|%Y-%m-%d}<br>Residual: %{y:.4f}',
), secondary_y=True)

# Plot ground truth maneuver lines
for _, row in ground_truth_df_test.iterrows():
    maneuver_date = row["Start_Timestamp"]
    fig.add_trace(go.Scatter(
        x=[maneuver_date, maneuver_date],
        y=[residuals_min, residuals_max],
        mode='lines',
        line=dict(color='green', dash='dash'),
        name='Ground Truth Maneuver',
        hoverinfo='text',
        text=[f"Maneuver Date: {maneuver_date.strftime('%Y-%m-%d')}"] * 2,
        showlegend=False
    ))
fig.add_trace(go.Scatter(
    x=[None],
    y=[None],
    mode='lines',
    line=dict(color='green', dash='dash'),
    name='Ground Truth Maneuver'
))
fig.add_hline(
    y=0,
    line_dash="dot",
    line_color="black",
    annotation_text="Zero Residual",
    annotation_position="top left",
    secondary_y=True
)
fig.update_layout(
    title='Observed vs Predicted with Residuals, Detected Anomalies, and Ground Truth Maneuvers',
    xaxis_title='Time',
    yaxis_title='Brouwer Mean Motion (scaled)',
    yaxis2_title='Residuals (Observed - Predicted)',
    template='plotly_white',
    width=1400,
    height=700,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    xaxis_range=[test_start, test_end]
)

fig.show()


### Residuals with detected anomaly and ground truth maneuver

In [ ]:

#Residual range for drawing maneuver lines 
residuals_min = df_result["residuals"].min()
residuals_max = df_result["residuals"].max()

# Create figure 
fig = go.Figure()

# Residuals
fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["residuals"],
    mode='lines+markers',
    name='Residuals',
    marker=dict(size=4),
    line=dict(color='blue')
))

#  Detected Anomalies
fig.add_trace(go.Scatter(
    x=df_result[df_result["anomaly"]].index,
    y=df_result[df_result["anomaly"]]["residuals"],
    mode='markers',
    name='Detected Anomalies',
    marker=dict(color='red', size=10, symbol='circle'),
    hovertemplate='Anomaly Date: %{x|%Y-%m-%d}<br>Residual: %{y:.4f}'
))

# Ground Truth Maneuver Lines
for _, row in ground_truth_df_test.iterrows():
    maneuver_date = row["Start_Timestamp"]
    fig.add_trace(go.Scatter(
        x=[maneuver_date, maneuver_date],
        y=[residuals_min, residuals_max],
        mode='lines',
        line=dict(color='green', dash='dash'),
        name='Ground Truth Maneuver',
        hoverinfo='text',
        text=[f"Maneuver Date: {maneuver_date.strftime('%Y-%m-%d')}"] * 2,
        showlegend=False
    ))
fig.add_trace(go.Scatter(
    x=[None],
    y=[None],
    mode='lines',
    line=dict(color='green', dash='dash'),
    name='Ground Truth Maneuver'
))
fig.add_hline(
    y=0,
    line_dash="dot",
    line_color="black",
    annotation_text="Zero Residual",
    annotation_position="top left"
)
fig.update_layout(
    title=dict(
        text='Residuals with Detected Anomalies and Ground Truth Maneuvers',
        font=dict(size=25)  
    ),
    xaxis=dict(
        title='Time(UTC)',
        titlefont=dict(size=25),  
        tickfont=dict(size=22)   
    ),
    yaxis=dict(
        title='Residuals (Observed - Predicted)',
        titlefont=dict(size=25),  
        tickfont=dict(size=22)    
    ),
    template='plotly_white',
    width=1400,
    height=600,
    legend=dict(
        font=dict(size=21),       
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    xaxis_range=[df_result.index.min(), df_result.index.max()]
)


fig.show()


## Evaluation Metrics
The code below for evaluation metrics was done for my own satisfaction. I'm not sure if it's appropriate to include it. In my report, I have not added this quantitative evaluation.

In [15]:

# Localize ground truth timestamps to UTC
ground_truth_df["Start_Timestamp"] = ground_truth_df["Start_Timestamp"].dt.tz_convert('UTC')
ground_truth_df["End_Timestamp"] = ground_truth_df["End_Timestamp"].dt.tz_convert('UTC')
buffer = timedelta(days=2)  
detected_anomalies = df_result[df_result["anomaly"]].index
test_start = df_result.index.min()
test_end = df_result.index.max()

filtered_gt_df = ground_truth_df[
    (ground_truth_df["Start_Timestamp"] >= test_start) &
    (ground_truth_df["Start_Timestamp"] <= test_end)
]
gt_intervals = list(zip(filtered_gt_df["Start_Timestamp"], filtered_gt_df["End_Timestamp"]))

#True Positives (TP) — Detected anomalies within buffer range of any interval
true_positives = 0
matched_anomalies = set()

for anomaly_time in detected_anomalies:
    for start, end in gt_intervals:
        if (start - buffer) <= anomaly_time <= (end + buffer):
            true_positives += 1
            matched_anomalies.add(anomaly_time)
            break

#False Positives (FP) — Detected anomalies outside all intervals
false_positives = len(detected_anomalies) - len(matched_anomalies)

#False Negatives (FN) — Intervals with no matching anomaly
false_negatives = 0
for start, end in gt_intervals:
    if not any((start - buffer) <= anomaly <= (end + buffer) for anomaly in detected_anomalies):
        false_negatives += 1

#Calculate metrics
precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) else 0
f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

#Print results
print(f" True Positives: {true_positives}")
print(f"False Positives: {false_positives}")
print(f"False Negatives: {false_negatives}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1_score:.2f}")


 True Positives: 14
False Positives: 6
False Negatives: 9
Precision: 0.70
Recall: 0.61
F1 Score: 0.65
